<a href="https://colab.research.google.com/github/kinshukpcs24-ux/AI_Lab_Programs/blob/main/1BM24CS137_Week8_FOL_Forward_Chaining/1BM24CS137_Week8_FOL_Forward_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import copy
import re

# --- 1. Helper Functions for Logic ---

def is_variable(x):
    """A variable is a lowercase string representing a placeholder."""
    return isinstance(x, str) and x.islower()

def is_constant(x):
    """A constant is an uppercase string or capitalized name."""
    return isinstance(x, str) and not x.islower()

def subst(theta, expr):
    """Applies a substitution theta (dict) to an expression (tuple or list)."""
    if isinstance(expr, tuple):
        return tuple(subst(theta, arg) for arg in expr)
    elif isinstance(expr, list):
        return [subst(theta, arg) for arg in expr]
    elif expr in theta:
        return subst(theta, theta[expr])
    else:
        return expr

# --- 2. Unification Algorithm ---

def unify(x, y, theta={}):
    """Unifies two expressions x and y under substitution theta."""
    if theta is None:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None
        # Unify predicate name and then arguments sequentially
        new_theta = unify(x[0], y[0], theta)
        for arg_x, arg_y in zip(x[1:], y[1:]):
            new_theta = unify(arg_x, arg_y, new_theta)
        return new_theta
    else:
        return None

def unify_var(var, x, theta):
    """Unifies a variable var with an expression x under theta."""
    if var in theta:
        return unify(theta[var], x, theta)
    elif x in theta:
        return unify(var, theta[x], theta)
    elif occur_check(var, x, theta):
        return None
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta

def occur_check(var, x, theta):
    """Checks if var occurs inside x under theta (to prevent infinite loops)."""
    if var == x:
        return True
    elif is_variable(x) and x in theta:
        return occur_check(var, theta[x], theta)
    elif isinstance(x, tuple):
        return any(occur_check(var, arg, theta) for arg in x)
    return False

# --- 3. Standardize Variables ---

_counter = 0
def standardize_variables(rule):
    """Renames all variables in a rule to unique names to avoid conflicts."""
    global _counter
    _counter += 1
    suffix = f"_{_counter}"

    # Collect variables
    vars_in_rule = set()
    def collect_vars(expr):
        if is_variable(expr):
            vars_in_rule.add(expr)
        elif isinstance(expr, tuple):
            for arg in expr[1:]:
                collect_vars(arg)

    for prem in rule['premises']:
        collect_vars(prem)
    collect_vars(rule['conclusion'])

    # Map original variables to standard unique ones
    rename_map = {v: f"{v}{suffix}" for v in vars_in_rule}

    standardized_premises = [subst(rename_map, p) for p in rule['premises']]
    standardized_conclusion = subst(rename_map, rule['conclusion'])

    return {'premises': standardized_premises, 'conclusion': standardized_conclusion}

# --- 4. FOL Forward Chaining Algorithm ---

def get_all_unifiers(premises, facts, theta):
    """Finds all substitutions theta that satisfy the premises given a set of facts."""
    if not premises:
        return [theta]

    p1 = premises[0]
    p1_sub = subst(theta, p1)
    results = []

    for fact in facts:
        theta_prime = unify(p1_sub, fact, theta.copy())
        if theta_prime is not None:
            # Recursively solve remaining premises with current substitution
            results.extend(get_all_unifiers(premises[1:], facts, theta_prime))

    return results

def fol_fc_ask(kb, alpha):
    """
    FOL-FC-ASK(KB, alpha) returns a substitution theta that unifies with alpha, or False.
    """
    rules = []
    facts = set()

    for item in kb:
        if isinstance(item, dict):
            rules.append(item)
        elif isinstance(item, tuple):
            facts.add(item)

    while True:
        new_facts = set()
        for rule in rules:
            std_rule = standardize_variables(rule)
            premises = std_rule['premises']
            conclusion = std_rule['conclusion']

            unifiers = get_all_unifiers(premises, list(facts), {})

            for theta in unifiers:
                q_prime = subst(theta, conclusion)

                already_exists = False
                for existing in facts.union(new_facts):
                    if unify(q_prime, existing, {}) is not None and unify(existing, q_prime, {}) is not None:
                        already_exists = True
                        break

                if not already_exists:
                    new_facts.add(q_prime)

                    phi = unify(q_prime, alpha, {})
                    if phi is not None:
                        return phi

        if not new_facts.intersection(facts) and len(new_facts) == 0:
            break

        facts.update(new_facts)

    return False

# --- 5. User Input Parsing Utility ---

def parse_predicate(text):
    """Parses a string like 'Owns(Nono, M1)' into ('Owns', 'Nono', 'M1'), or zero-arg proposition 'Rain' into ('Rain',)"""
    text = text.strip().replace(" ", "")

    # Try matching zero-argument predicate written as 'Predicate()' or just 'Predicate'
    if text.endswith("()"):
        return (text[:-2],)
    if "(" not in text and ")" not in text:
        return (text,)

    match = re.match(r"^([a-zA-Z0-9_]+)\(([^)]+)\)$", text)
    if not match:
        raise ValueError(f"Invalid predicate format: '{text}'. Must be like Predicate(arg1, arg2) or Predicate")
    pred_name = match.group(1)
    args = match.group(2).split(",")
    return (pred_name,) + tuple(args)

def parse_input_line(line):
    """Parses a line into either a fact tuple or a rule dict"""
    line = line.strip()
    if not line:
        return None

    if "=>" in line:
        # It is a rule: Premise1 & Premise2 => Conclusion
        parts = line.split("=>")
        premises_part = parts[0].strip()
        conclusion_part = parts[1].strip()

        premises_list = [parse_predicate(p) for p in premises_part.split("&")]
        conclusion = parse_predicate(conclusion_part)
        return {"premises": premises_list, "conclusion": conclusion}
    else:
        # It is a single fact
        return parse_predicate(line)

# --- 6. Main Execution Block with Interactive Input ---

def main():
    print("=== Interactive First Order Logic Forward Chaining ===\n")

    kb = []

    print("STEP 1: Enter your RULES (one per line). Enter an empty line when finished:")
    while True:
        try:
            user_line = input("Rule > ").strip()
            if not user_line:
                break
            if "=>" not in user_line:
                print("Warning: This line does not look like a rule (missing '=>'). Enter rules here, or leave empty to move to facts.")
                continue
            parsed = parse_input_line(user_line)
            if parsed:
                kb.append(parsed)
        except Exception as e:
            print(f"Error parsing rule: {e}. Please try again.")

    print("\nSTEP 2: Enter your FACTS (one per line). Enter an empty line when finished:")
    while True:
        try:
            user_line = input("Fact > ").strip()
            if not user_line:
                break
            if "=>" in user_line:
                print("Warning: Rules are not allowed in this step. Please enter facts only (no '=>').")
                continue
            parsed = parse_input_line(user_line)
            if parsed:
                kb.append(parsed)
        except Exception as e:
            print(f"Error parsing fact: {e}. Please try again.")

    if not kb:
        print("Knowledge base is empty. Exiting...")
        return

    print(f"\nSuccessfully registered {len(kb)} KB elements (Rules and Facts combined).")

    while True:
        try:
            query_str = input("\nSTEP 3: Enter your QUERY (e.g., Criminal(x) or Criminal(West)): ").strip()
            if not query_str:
                continue
            query = parse_predicate(query_str)
            break
        except Exception as e:
            print(f"Error parsing query: {e}. Please try again.")

    print(f"Executing Forward Chaining query... Ask({query})")
    result = fol_fc_ask(kb, query)

    if result is not False:
        print(f"\nResult: Success! Found substitution: {result}")
    else:
        print("\nResult: False / No valid proof found under this Knowledge Base.")

if __name__ == "__main__":
    main()

=== Interactive First Order Logic Forward Chaining ===

STEP 1: Enter your RULES (one per line). Enter an empty line when finished:
Rule > HasFever(x) & HasSoreThroat(x) => HasSignsOfInfection(x)
Rule > HasSignsOfInfection(x) & HasMuscleAches(x) => TestForInfluenza(x)
Rule > 

STEP 2: Enter your FACTS (one per line). Enter an empty line when finished:
Fact >  HasFever(P)
Fact > HasSoreThroat(P)
Fact > HasMuscleAches(P)
Fact > 

Successfully registered 5 KB elements (Rules and Facts combined).

STEP 3: Enter your QUERY (e.g., Criminal(x) or Criminal(West)): TestForInfluenza(x)
Executing Forward Chaining query... Ask(('TestForInfluenza', 'x'))

Result: Success! Found substitution: {'x': 'P'}
